In [0]:
%pip install plotly

In [0]:
# ============================================
# ACTIVIDAD 4.2 — Estadística básica y Plotly
# Reto FEMSA: conciliación de volúmenes
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

# Pregunta de análisis:
# ¿Cómo varía la brecha absoluta entre PTAR calculada
# y el volumen publicado en Reporting según el mes y la planta?
#
# Unidad: litros (L).
# Política de faltantes: se eliminan únicamente filas
# sin PTAR o Reporting; no se usan Excel originales.

# 1. Ubicar el archivo CSV procesado del reto
notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

workspace_notebook_path = Path("/Workspace") / notebook_path.lstrip("/")
repo_root = workspace_notebook_path.parents[2]

fuente = (
    repo_root
    / "assignments"
    / "reto_femsa"
    / "data"
    / "processed"
    / "etapa2_conciliacion_sql.csv"
)

print("Ruta del notebook:", workspace_notebook_path)
print("Fuente procesada:", fuente)
print("¿Archivo encontrado?:", fuente.exists())

# 2. Leer el CSV procesado
raw = pd.read_csv(fuente)

print("\nColumnas originales:")
print(raw.columns.tolist())
print("\nRegistros originales:", len(raw))

# 3. Elegir variables de análisis
# Nota: en este archivo el campo se llama ptar_l, no PTAR.
df = raw[["planta_id", "mes", "ptar_l", "reporting_L"]].copy()

df = df.rename(columns={
    "reporting_L": "reporting_l"
})

df["ptar_l"] = pd.to_numeric(df["ptar_l"], errors="coerce")
df["reporting_l"] = pd.to_numeric(df["reporting_l"], errors="coerce")

print("\nValores faltantes antes de limpiar:")
print(df[["ptar_l", "reporting_l"]].isna().sum())

# Eliminar solamente filas que no pueden compararse
df = df.dropna(subset=["ptar_l", "reporting_l"]).copy()

# Anonimizar identificador de planta
df["planta_anon"] = (
    "Planta_" + df["planta_id"].astype(int).astype(str).str.zfill(2)
)

# Indicador central
df["brecha_absoluta_l"] = (df["ptar_l"] - df["reporting_l"]).abs()

print("\nRegistros después de limpiar:", len(df))
print("\nMuestra de datos:")
display(
    df[["planta_anon", "mes", "ptar_l", "reporting_l", "brecha_absoluta_l"]]
    .head()
)

# 4. Estadística descriptiva de la brecha
serie = df["brecha_absoluta_l"]

estadisticas = pd.DataFrame({
    "métrica": [
        "número de observaciones",
        "media",
        "mediana",
        "moda",
        "desviación estándar",
        "varianza",
        "mínimo",
        "Q1",
        "Q3",
        "máximo"
    ],
    "valor_litros": [
        serie.count(),
        serie.mean(),
        serie.median(),
        serie.mode().iloc[0] if not serie.mode().empty else np.nan,
        serie.std(),
        serie.var(),
        serie.min(),
        serie.quantile(0.25),
        serie.quantile(0.75),
        serie.max()
    ]
})

print("\nEstadística descriptiva de la brecha absoluta:")
display(estadisticas.style.format({"valor_litros": "{:,.2f}"}))

# 5. Resumen mensual
resumen_mensual = (
    df.groupby("mes", as_index=False)
    .agg(
        ptar_total_l=("ptar_l", "sum"),
        reporting_total_l=("reporting_l", "sum"),
        brecha_promedio_l=("brecha_absoluta_l", "mean"),
        registros=("brecha_absoluta_l", "count")
    )
)

print("\nResumen mensual:")
display(
    resumen_mensual.style.format({
        "ptar_total_l": "{:,.0f}",
        "reporting_total_l": "{:,.0f}",
        "brecha_promedio_l": "{:,.2f}"
    })
)

# 6. Gráfica 1: histograma
fig_hist = px.histogram(
    df,
    x="brecha_absoluta_l",
    nbins=25,
    title="Distribución de la brecha absoluta entre PTAR y Reporting",
    labels={
        "brecha_absoluta_l": "Brecha absoluta (L)",
        "count": "Número de registros"
    },
    template="plotly_white",
    color_discrete_sequence=["#007E87"]
)

fig_hist.update_layout(title_x=0.5, bargap=0.08)
fig_hist.show()

# 7. Gráfica 2: comparación por mes
fig_box = px.box(
    df,
    x="mes",
    y="brecha_absoluta_l",
    points="all",
    title="Variación de la brecha absoluta por mes",
    labels={
        "mes": "Mes",
        "brecha_absoluta_l": "Brecha absoluta (L)"
    },
    template="plotly_white",
    color_discrete_sequence=["#00A6A6"]
)

fig_box.update_layout(title_x=0.5)
fig_box.show()

# 8. Conclusión inicial automática
mes_critico = resumen_mensual.loc[
    resumen_mensual["brecha_promedio_l"].idxmax(),
    "mes"
]

brecha_media = serie.mean()

print(f"""
CONCLUSIÓN INICIAL
La brecha absoluta promedio es de {brecha_media:,.2f} L.
El mes {int(mes_critico)} presenta la mayor brecha promedio.
Esto permite priorizar la revisión de conciliaciones en los
periodos con mayor diferencia entre PTAR y Reporting.
""")